# C11-neural-training — Practice p13 — Solution


**Type:** proof / derivation · **Difficulty:** core · **Concepts:** batch-normalization


All featurewise sums below are over batch axis 0 and retain shape \((D,)\).
First \(d\beta=\sum_iG_i\), \(d\gamma=\sum_iG_i\widehat X_i\), and
\(d\widehat X=G\gamma\). Put \(q=X-\mu\). The indexed backward chain is
\[
ds=\sum_i d\widehat X_i q_i,\qquad
dv=-\tfrac12 ds\,(v+\epsilon)^{-3/2},
\]
\[
dq_i=s\,d\widehat X_i+\frac{2q_i}{N}dv,\qquad
d\mu=-\sum_i dq_i,\qquad dX_i=dq_i+\frac{d\mu}{N}.
\]
Substituting \(q=s^{-1}\widehat X\), \(\sum_iq_i=0\), and
\(d\widehat X_i=\gamma G_i\), then collecting the direct, variance, and mean
paths gives
\[
dX=\frac{\gamma s}{N}\left(NG-\sum_iG_i-
\widehat X\sum_i(G_i\widehat X_i)\right),
\]
of shape \((N,D)\). Because \(\sum_i\widehat X_i=0\), summing gives
\(\sum_i dX_i=0\). The centered moment is exactly
\(\sum_i\widehat X_i dX_i=\gamma s\,\sum_i(G_i\widehat X_i)
\epsilon/(v+\epsilon)\), so it vanishes when \(\epsilon=0\) and is small up to
epsilon's stabilizing effect.

Training normalizes the current batch with denominator \(N\); PyTorch updates
the running variance with the corresponding unbiased \(N-1\) estimate.
Gamma and beta are optimizer-trained parameters. Running mean/variance are
module-updated buffers: train mode uses batch statistics and updates them,
whereas eval mode uses fixed running statistics and does not update them.


In [ ]:
import numpy as np
rng_p13=np.random.default_rng(13)
X_p13=rng_p13.normal(size=(5,3)); G_p13=rng_p13.normal(size=(5,3)); gamma_p13=np.array([1.2,.7,2.])
eps_p13=1e-5; mu_p13=X_p13.mean(0); centered_p13=X_p13-mu_p13; v_p13=np.mean(centered_p13**2,0)
s_p13=(v_p13+eps_p13)**-.5; xhat_p13=centered_p13*s_p13
dbeta_p13=G_p13.sum(0); dgamma_p13=(G_p13*xhat_p13).sum(0)
dX_p13=(gamma_p13*s_p13/5)*(5*G_p13-G_p13.sum(0)-xhat_p13*(G_p13*xhat_p13).sum(0))


### Answer check


In [ ]:
assert dbeta_p13.shape == dgamma_p13.shape == (3,) and dX_p13.shape == X_p13.shape
assert np.allclose(dX_p13.sum(0), 0.0, atol=1e-14, rtol=0.0)
moment_p13=(xhat_p13*dX_p13).sum(0)
expected_moment_p13=gamma_p13*s_p13*(G_p13*xhat_p13).sum(0)*eps_p13/(v_p13+eps_p13)
assert np.allclose(moment_p13, expected_moment_p13, atol=1e-13, rtol=1e-10)
